In [97]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torch
from torch.utils.data import Dataset
from torchvision.io import decode_image
from pathlib import Path
import zipfile


In [98]:
class CustomImageDataset(Dataset):
    def __init__(self, data_folder: Path, transform=None):
        self.data_folder = Path(data_folder)
        self.image_dir = self.data_folder / "images"
        self.label_dir = self.data_folder / "labels"

        self.images = sorted(self.image_dir.glob("*.jpg"))
        self.transform = transform

        # Verify image -> annotation correspondence
        missing_labels = [
            img.name
            for img in self.images
            if not (self.label_dir / f"{img.stem}.txt").exists()
        ]

        if missing_labels:
            raise RuntimeError(
                f"{len(missing_labels)} images have no annotation file. "
                f"Example: {missing_labels[:5]}"
            )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]

        image = decode_image(str(img_path))

        label_path = self.label_dir / f"{img_path.stem}.txt"

        labels = []

        with label_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                # YOLO:
                # class_id x_center y_center width height
                values = [float(x) for x in line.split()]
                labels.append(values)

        if labels:
            target = torch.tensor(labels, dtype=torch.float32)
        else:
            target = torch.empty((0, 5), dtype=torch.float32)

        # For detection, transform image + boxes together
        if self.transform:
            image, target = self.transform(image, target)

        return image, target

In [92]:
data_dir = Path("./data")

challenge_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-test-challenge")
test_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-test-dev")
train_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-train")
val_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-val")

In [99]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
train_path = Path("./VisDrone2019-DET-train.zip")
test_path = Path("./VisDrone2019-DET-test-dev.zip")
val_path = Path("./VisDrone2019-DET-val.zip")
challenge_path = Path("./VisDrone2019-DET-test-challange.zip")

data_dir = Path("./data")

data_dir.mkdir(exist_ok=True)

print("Extraction started")
for path_file in [train_path, test_path, val_path]:
    with zipfile.ZipFile(path_file, "r") as zip_ref:
        zip_ref.extractall(data_dir)

print(f"Extracted to {data_dir.resolve()}")


In [12]:
sample_path = train_path / "images" / "0000002_00005_d_0000014.jpg"

In [11]:
CLASS_MAPPING = {
    0: "ignored_regions",
    1: "pedestrian",
    2: "people",
    3: "bicycle",
    4: "car",
    5: "van",
    6: "truck",
    7: "tricycle",
    8: "awning_tricycle",
    9: "bus",
    10: "motor",
    11: "others",
}

In [5]:
with Image.open(str(sample_path)) as im:
    im.show()

In [6]:
data = pd.read_csv(data_dir / "VisDrone2019-DET-train" / "annotations" / "0000002_00005_d_0000014.txt", header=None)

In [7]:
data

,0,1,2,3,4,5,6,7
0,684,8,273,116,0,0,0,0
1,406,119,265,70,0,0,0,0
2,255,22,119,128,0,0,0,0
3,1,3,209,78,0,0,0,0
4,708,471,74,33,1,4,0,1
...,...,...,...,...,...,...,...,...
83,136,119,6,8,1,10,0,1
84,841,487,21,26,1,10,0,0
85,912,139,34,73,0,0,0,0
86,223,345,8,11,1,2,0,2


In [22]:
def convert_to_yolo_format(txt_path: Path):
    output_dir = txt_path.parent.parent / "labels"
    output_dir.mkdir(parents=True, exist_ok=True)

    converted_lines = []

    with txt_path.open("r") as f:
        for line in f:
            lst = line.strip().split(",")

            # class_id x_center y_center width height
            converted_line = " ".join([lst[5], *lst[:4]])
            converted_lines.append(converted_line)

    output_path = output_dir / txt_path.name

    with output_path.open("w") as f:
        f.write("\n".join(converted_lines))

In [28]:
for data_set in list(data_dir.iterdir()):
    for file_path in (data_set / "annotations/").glob("*.txt"):
        convert_to_yolo_format(file_path)

IsADirectoryError: [Errno 21] Is a directory: './data/VisDrone2019-DET-test-challenge/annotations'